# LSTM (Long Short-Term Memory)

Implementar un modelo usando LSTM (secuencias): dataset de series temporales (p. ej. precios, sensor data) o texto secuencial (modelado de lenguaje char-level o word-level). 



In [ ]:
# SEMANA 4 - LSTM Y SERIES TEMPORALES
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# GENERANDO DATOS DE PRUEBA (SERIE TEMPORAL)
sin_wave = np.sin(np.linspace(0, 100, 1000))

# CREAR SECUENCIAS PARA EL ENTRENAMIENTO
def create_sequences(data, seq_length):
    X, Y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        Y.append(data[i+seq_length])
    return np.array(X), np.array(Y)

seq_length = 50
X, Y = create_sequences(sin_wave, seq_length)

# AJUSTAR EMISION DE DIMENSIONES PARA LSTM (samples, timesteps, features)
X = np.expand_dims(X, axis=-1)

# DIVISION DE ENTRENAMIENTO Y TEST
split = int(0.8 * len(X))
X_train, X_test = X[:split], X[split:]
Y_train, Y_test = Y[:split], Y[split:]

# MODELO LSTM PARA SERIES TEMPORALES
model = Sequential([
    LSTM(64, input_shape=(seq_length, 1), return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.fit(X_train, Y_train, validation_data=(X_test, Y_test), epochs=10, batch_size=32)

# EVALUACION DEL MODELO
loss, mae = model.evaluate(X_test, Y_test)
print(f'Loss (MSE): {loss:.4f}, MAE: {mae:.4f}')

# PREDICCIONES
Y_pred = model.predict(X_test)

# GRAFICAR RESULTADOS
plt.figure(figsize=(10, 5))
plt.plot(Y_test, label='Real', color='blue')
plt.plot(Y_pred, label='Prediccion', color='orange')
plt.title('Prediccion de Serie Temporal con LSTM')
plt.xlabel('Tiempo')
plt.ylabel('Valor')
plt.legend()
plt.show()

# ANALISIS DE ERRORES EN LAS PREDICCIONES
errors = np.abs(Y_test - Y_pred.flatten())
max_error_indices = np.argsort(errors)[-5:]

print("Analisis de los mayores errores cometidos:")
print("-" * 40)
for idx in max_error_indices:
    print(f"Indice: {idx} | Valor Real: {Y_test[idx]:.4f} | Prediccion: {Y_pred[idx][0]:.4f} | Error Absoluto: {errors[idx]:.4f}")